# 🚀 **ColabRP Backend: KoboldCPP + Cloudflare (Solo Texto)**

Este notebook despliega un backend **ligero** para Roleplay en Google Colab T4. 
Si buscas generar imágenes, usa el notebook `image_backend.ipynb`.

In [1]:
# @title 1. Configuración Inicial

# @markdown ### 🧠 Modelo de Texto (LLM)
MODEL_URL = "https://huggingface.co/Lewdiculous/L3-8B-Stheno-v3.2-GGUF-IQ-Imatrix/resolve/main/L3-8B-Stheno-v3.2-Q6_K-imat.gguf" # @param ["https://huggingface.co/Lewdiculous/L3-8B-Stheno-v3.2-GGUF-IQ-Imatrix/resolve/main/L3-8B-Stheno-v3.2-Q6_K-imat.gguf", "https://huggingface.co/TheBloke/MythoMax-L2-13B-GGUF/resolve/main/mythomax-l2-13b.Q4_K_M.gguf", "https://huggingface.co/Sao10K/Fimbulvetr-11B-v2-GGUF/resolve/main/Fimbulvetr-11B-v2.q5_k_m.gguf", "https://huggingface.co/brittlewis12/Kunoichi-DPO-v2-7B-GGUF/resolve/main/kunoichi-dpo-v2-7b.Q8_0.gguf"]
MODEL_NAME = "model.gguf"

# @markdown ### ⚙️ Rendimiento T4 (Contexto y VRAM)
CONTEXT_SIZE = 12288 # @param [8192, 12288, 16384, 24576]
GPU_LAYERS = 99 # @param {type:"integer"}
STARTUP_TIMEOUT = 600 # @param {type:"integer"}
AUTO_T4_FALLBACK = True # @param {type:"boolean"}
KOBOLD_EXTRA_ARGS = "--highpriority" # @param {type:"string"}

# @markdown ### ☁️ Sincronización (npoint.io)
NPOINT_ID = "" # @param {type:"string"}

import time
import requests
import subprocess
import re
import socket
import shlex

def log(msg):
    print(f"[ColabRP] {msg}")

In [2]:
# @title 2. Instalar Herramientas
log("Instalando dependencias base...")
!apt-get update && apt-get install -y aria2

log(f"Descargando LLM: {MODEL_URL}...")
!aria2c -x 16 -s 16 -o {MODEL_NAME} {MODEL_URL}

log("Descargando KoboldCPP...")
!curl -L -o koboldcpp https://github.com/LostRuins/koboldcpp/releases/latest/download/koboldcpp-linux-x64
!chmod +x koboldcpp

log("Descargando Cloudflared...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
# @title 3. Ejecutar Sistema

ACTIVE_CONTEXT = None
ACTIVE_GPULAYERS = None

def wait_for_port_or_exit(process, port, timeout_s):
    log(f"Esperando a que el puerto {port} esté abierto (Modelo cargando...)")
    start_time = time.time()
    while True:
        if process.poll() is not None:
            log(f"❌ KoboldCPP cerró antes de quedar listo (exit={process.returncode}).")
            return False

        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        sock.close()

        if result == 0:
            log(f"✅ Puerto {port} abierto. KoboldCPP está listo.")
            return True

        if time.time() - start_time > timeout_s:
            log("⚠️ Timeout esperando KoboldCPP.")
            return False

        time.sleep(2)

def stop_process(process):
    if process and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()

def build_startup_attempts():
    attempts = [(int(CONTEXT_SIZE), int(GPU_LAYERS))]

    if AUTO_T4_FALLBACK:
        if int(CONTEXT_SIZE) > 12288:
            attempts.append((12288, min(int(GPU_LAYERS), 80)))
        if int(CONTEXT_SIZE) > 8192:
            attempts.append((8192, min(int(GPU_LAYERS), 60)))
        elif int(GPU_LAYERS) > 80:
            attempts.append((int(CONTEXT_SIZE), 80))

    unique_attempts = []
    for item in attempts:
        if item not in unique_attempts:
            unique_attempts.append(item)

    return unique_attempts

def launch_kobold(context_size, gpu_layers):
    cmd = [
        "./koboldcpp",
        "--model", MODEL_NAME,
        "--contextsize", str(context_size),
        "--gpulayers", str(gpu_layers),
        "--usecublas",
        "--port", "5001",
        "--quiet",
    ]

    extra_args = KOBOLD_EXTRA_ARGS.strip()
    if extra_args:
        cmd.extend(shlex.split(extra_args))

    log(f"Comando: {' '.join(cmd)}")
    return subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

def start_kobold():
    global ACTIVE_CONTEXT, ACTIVE_GPULAYERS
    attempts = build_startup_attempts()

    for idx, (ctx, gpu) in enumerate(attempts, start=1):
        log(f"Iniciando KoboldCPP (intento {idx}/{len(attempts)}) con ctx={ctx}, gpulayers={gpu}...")
        process = launch_kobold(ctx, gpu)

        if wait_for_port_or_exit(process, 5001, int(STARTUP_TIMEOUT)):
            ACTIVE_CONTEXT = ctx
            ACTIVE_GPULAYERS = gpu
            log(f"✅ Backend listo con ctx={ctx}, gpulayers={gpu}.")
            return True

        stop_process(process)
        log("Reintentando con configuración más conservadora para T4...")

    log("❌ No se pudo iniciar KoboldCPP con las configuraciones probadas.")
    return False

def start_tunnel():
    log("Iniciando túnel Cloudflare...")
    process = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:5001"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    url_found = False
    regex = r"https://[a-zA-Z0-9-]+\.trycloudflare\.com"

    while True:
        line = process.stdout.readline()
        if not line and process.poll() is not None:
            break

        match = re.search(regex, line)
        if match and not url_found:
            url = match.group(0)
            log(f"¡Túnel activo! URL: {url}")
            update_npoint(url)
            url_found = True

    if not url_found:
        log("⚠️ Cloudflared terminó sin publicar URL.")

def update_npoint(url):
    global NPOINT_ID
    if not NPOINT_ID:
        return

    if "npoint.io" in NPOINT_ID:
        NPOINT_ID = NPOINT_ID.rstrip("/").split("/")[-1]

    target_url = f"https://api.npoint.io/{NPOINT_ID}"

    # 1. LEER datos actuales para no borrar otras configuraciones (Merge)
    current_data = {}
    try:
        resp = requests.get(target_url, timeout=15)
        if resp.status_code == 200:
            current_data = resp.json()
            if not isinstance(current_data, dict):
                # Si por algún motivo era un string o array, lo forzamos a dict para arreglarlo
                current_data = {}
    except Exception as e:
        log(f"⚠️ Error leyendo npoint anterior (se creará nuevo): {e}")

    # 2. ACTUALIZAR solo la api_url (Texto)
    current_data["api_url"] = url
    current_data["llm_model_url"] = MODEL_URL
    current_data["llm_context"] = ACTIVE_CONTEXT
    current_data["llm_gpulayers"] = ACTIVE_GPULAYERS
    current_data["updated_at_llm"] = time.time()

    # 3. GUARDAR
    try:
        requests.post(target_url, json=current_data, timeout=15)
        log(f"✅ URL actualizada en npoint.io/{NPOINT_ID}")
    except Exception as e:
        log(f"❌ Error actualizando npoint: {e}")

if start_kobold():
    start_tunnel()
else:
    log("Abortando túnel porque KoboldCPP no inició correctamente.")